# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [ ]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'daily'
DATA_PATH = DATA_DIR / '*.csv'

csv_files = list(DATA_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(
        f'Dataset tidak ditemukan di: {DATA_DIR}. '
        'Pastikan data telah diekstrak ke data/raw/daily/.'
    )

total_size_mb = sum(f.stat().st_size for f in csv_files) / 1_000_000
print(f'Folder Dataset: {DATA_DIR.name}')
print(f'Jumlah file: {len(csv_files):,} file CSV emiten')
print(f'Total ukuran file mentah: {total_size_mb:.1f} MB')

Folder Dataset: daily
Jumlah file: 824 file CSV emiten
Total ukuran file mentah: 83.5 MB


## Profil Polars

`scan_csv` dengan wildcard hanya membangun rencana query; data baru dibaca secara efisien dan paralel ketika `collect()` dipanggil.

In [ ]:
lazy_data = pl.scan_csv(
    str(DATA_PATH),
    schema_overrides={
        'open': pl.Float64,
        'low': pl.Float64,
        'high': pl.Float64,
        'close': pl.Float64,
        'volume': pl.Float64,
    },
    ignore_errors=True,
    include_file_paths='file_path',
).with_columns(
    pl.col('file_path')
    .str.split('/')
    .list.last()
    .str.split('\\')
    .list.last()
    .str.replace(r'\.csv$', '')
    .alias('ticker')
).drop('file_path')

print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

Schema({'timestamp': String, 'open': Float64, 'low': Float64, 'high': Float64, 'close': Float64, 'volume': Float64, 'ticker': String})


jumlah_baris,timestamp_null,open_null,low_null,high_null,close_null,volume_null,ticker_null
u32,u32,u32,u32,u32,u32,u32,u32
2469667,0,0,0,0,0,0,0


In [ ]:
contoh_data = lazy_data.head(10).collect()
contoh_data

timestamp,open,low,high,close,volume,ticker
str,f64,f64,f64,f64,f64,str
"""2001-04-16""",524.0,524.0,547.0,547.0,342131.0,"""AALI"""
"""2001-04-17""",547.0,524.0,571.0,524.0,345804.0,"""AALI"""
"""2001-04-18""",547.0,524.0,547.0,524.0,230361.0,"""AALI"""
"""2001-04-19""",524.0,500.0,524.0,500.0,457574.0,"""AALI"""
"""2001-04-20""",476.0,452.0,476.0,452.0,228262.0,"""AALI"""
"""2001-04-23""",452.0,452.0,452.0,452.0,11544.0,"""AALI"""
"""2001-04-24""",476.0,471.0,476.0,471.0,3148.0,"""AALI"""
"""2001-04-25""",476.0,471.0,485.0,471.0,345804.0,"""AALI"""
"""2001-04-26""",476.0,471.0,485.0,471.0,345804.0,"""AALI"""


## Profil DuckDB

DuckDB membaca file CSV langsung dari disk secara paralel dan menghasilkan ringkasan statistik setiap kolom.

In [ ]:
connection = duckdb.connect()
duckdb_path = str(DATA_PATH).replace('\\', '/')
profil_duckdb = connection.execute(
    'SUMMARIZE SELECT * FROM read_csv_auto(?)', [duckdb_path]
).pl()
profil_duckdb

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""timestamp""","""DATE""","""2001-04-16""","""2023-01-06""",4150,"""2014-04-30 13:09:01.969261""",null,"""2009-06-28""","""2015-04-28""","""2019-10-03""",2469667,0.00
"""open""","""BIGINT""","""0""","""17500000""",9928,"""13854.119691440183""","""315497.42820783606""","""123""","""314""","""923""",2469667,0.00
"""low""","""BIGINT""","""0""","""17500000""",9331,"""13733.68785427347""","""312561.0266281627""","""120""","""309""","""907""",2469667,0.00
"""high""","""BIGINT""","""0""","""17500000""",10027,"""14051.004636252579""","""320917.6713742967""","""125""","""319""","""938""",2469667,0.00
"""close""","""BIGINT""","""1""","""17500000""",9887,"""13945.465129509363""","""318360.52224509616""","""123""","""314""","""923""",2469667,0.00
"""volume""","""BIGINT""","""0""","""26555346900""",598737,"""12418529.026065053""","""88123738.28316604""","""0""","""46137""","""2428576""",2469667,0.00


## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.

## Catatan Milestone 1

Dari hasil profiling awal menggunakan Polars dan DuckDB, beberapa poin penting dari data ini antara lain sebagai berikut:

1. **Ukuran Data dan Jumlah Baris**
   - Dataset ini diambil dari folder `daily` yang berisi **824 file CSV** (tiap file mewakili satu kode saham di BEI).
   - Total data mencapai **2.469.667 baris** dengan ukuran total sekitar **83,5 MB**.

2. **Tipe dan Struktur Kolom**
   - Ada 7 kolom utama: `timestamp` (tanggal transaksi dari tahun 2001 sampai 2023), `ticker` (kode saham hasil ekstraksi nama file), serta pergerakan harga/transaksi yaitu `open`, `low`, `high`, `close`, dan `volume`.
   - Kolom harga dan volume sudah bertipe numerik, tapi `timestamp` masih berupa string/teks sehingga nantinya perlu di-parse ke tipe Date.

3. **Pengecekan Missing Values**
   - Hasil profiling menunjukkan **tidak ada nilai null sama sekali (0%)** di seluruh kolom.
   - Meskipun tidak ada null secara teknis, terdapat cukup banyak nilai `0` pada harga dan volume yang perlu diperhatikan karena bisa jadi data saham tidak aktif.

4. **Anomali yang Perlu Ditangani di Milestone 2**
   - **Harga bernilai 0:** Nilai minimum untuk `open`, `low`, dan `high` tercatat 0. Di bursa reguler harga saham umumnya punya batas bawah, jadi nilai 0 ini kemungkinan saham sedang disuspensi atau ada error saat pencatatan data lama.
   - **Outlier harga yang sangat ekstrem:** Nilai median harga saham ada di 314 dan 75% datanya di bawah 923, tapi nilai maksimalnya bisa tembus sampai 17.500.000. Distribusinya sangat jomplang ke kanan, jadi di Milestone 2 perlu dibersihkan dengan metode IQR dan Winsorization (clipping).
   - **Volume transaksi 0:** Kuartil pertama (Q25) untuk volume bernilai 0, artinya lebih dari seperempat data tidak memiliki aktivitas transaksi perdagangan (saham tidur atau hari libur).
